# Stochastic Calculus from Zero · Episode 15: The Itô-Doeblin formula, properly

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **No data.** Every path is simulated with a fixed seed. The share is illustrative: 100, volatility 20%, real-world drift 8% a year.
>
> Educational material only, not financial advice.

1. Setup · 2. Last episode's answer · 3. Taylor plus the multiplication table · 4. Checked on a path · 5. Episode 1, solved · 6. The log price · 7. History · 8. This episode's question · 9. Exercises · 10. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import itertools
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

out = {"meta": {
    "series": "Stochastic Calculus from Zero",
    "episode": 15,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}

The building blocks used throughout the series (`episodes/_shared/stochastic.py`):

In [ ]:
# Building blocks for Stochastic Calculus from Zero, written out by hand so every step can be read. Copied verbatim
# into each notebook by make_notebook.py, so every notebook runs on its own. Needs: itertools, numpy as np.
#   Coin world (Part 1): a share at 100 that each toss multiplies by 1.2 (heads) or 0.8 (tails), three tosses.
#   Real-world chance of heads 0.6; a deposit grows 2% a step, so the risk-neutral chance is 0.55. Made up for
#   teaching.

class CoinWorld:
    """Every possible world of a few coin tosses, with real-world (P) and risk-neutral (Q) weights.

    A world is a string such as "HTH". Its first n letters are what you know after n tosses, so the block of
    information at time n that a world sits in is simply world[:n].
    """

    def __init__(self, s0=100.0, up=1.2, down=0.8, rate=0.02, p=0.6, steps=3):
        self.s0, self.up, self.down, self.rate, self.p, self.steps = s0, up, down, rate, p, steps
        self.q = (1 + rate - down) / (up - down)   # the odds that make the discounted share a martingale
        self.worlds = ["".join(w) for w in itertools.product("HT", repeat=steps)]

    def weight(self, world, measure="P"):
        heads = self.p if measure == "P" else self.q
        return float(np.prod([heads if c == "H" else 1 - heads for c in world]))

    def path(self, world):
        """Share prices S0, S1, ..., one per toss, along this world."""
        s = [self.s0]
        for c in world:
            s.append(s[-1] * (self.up if c == "H" else self.down))
        return s

    def price(self, world, n):
        return self.path(world)[n]

    def expect(self, f, measure="P"):
        return sum(self.weight(w, measure) * f(w) for w in self.worlds)

    def given(self, n, f, measure="P", by=None):
        """E[f | information at time n], as {block: best guess}. `by` groups worlds some other way (e.g. by price)."""
        key = by or (lambda w: w[:n])
        blocks = {}
        for w in self.worlds:
            blocks.setdefault(key(w), []).append(w)
        return {b: sum(self.weight(w, measure) * f(w) for w in ws) / sum(self.weight(w, measure) for w in ws)
                for b, ws in blocks.items()}

    def table(self, **extra):
        """One row per world (HHH ... TTT): weights, prices S0..Sn, number of heads, plus extra columns
        given as functions of the world, e.g. table(call=lambda w: max(cw.price(w, 3) - 100, 0))."""
        rows = []
        for w in self.worlds:
            row = {"world": w, "P": self.weight(w, "P"), "Q": self.weight(w, "Q"), "heads": w.count("H")}
            row |= {f"S{n}": s for n, s in enumerate(self.path(w))}
            row |= {k: f(w) for k, f in extra.items()}
            rows.append(row)
        return rows

    def blocks(self, n):
        """The blocks of information at time n: lists of worlds you cannot yet tell apart."""
        out = {}
        for w in self.worlds:
            out.setdefault(w[:n], []).append(w)
        return list(out.values())


def brownian_paths(n_paths, steps, years, rng):
    """n_paths Brownian paths on a grid of `steps` equal steps: returns (t, W, dW), W starting at 0."""
    dt = years / steps
    dW = rng.standard_normal((n_paths, steps)) * np.sqrt(dt)
    W = np.concatenate([np.zeros((n_paths, 1)), dW.cumsum(axis=1)], axis=1)
    return np.linspace(0, years, steps + 1), W, dW


def sum_of_squares(increments):
    """Sum of squared steps along each path (the quadratic variation on this grid)."""
    return (np.asarray(increments) ** 2).sum(axis=-1)


def chain_rule_sum(W, dW):
    """The school chain rule's prediction for W_T^2: add up 2 W dW, with W taken at the start of each step."""
    return 2 * (W[..., :-1] * dW).sum(axis=-1)


# ---- Part 2 additions -------------------------------------------------------------------------------------------
def riemann_sums(W, dW):
    """Sum of W dW sampled at the left end, the middle and the right end of each step (one path)."""
    return {"left": float((W[:-1] * dW).sum()), "mid": float((0.5 * (W[:-1] + W[1:]) * dW).sum()),
            "right": float((W[1:] * dW).sum())}


def ito_check(f, f1, f2, W, years=1.0):
    """For one path W on an even grid: the exact change f(W_T) - f(W_0), and what the school chain rule
    (sum of f'(W) dW) and the Ito formula (that plus 1/2 sum f''(W) dt) say it is."""
    dW, dt = np.diff(W), years / (len(W) - 1)
    chain = float((f1(W[:-1]) * dW).sum())
    return {"exact": float(f(W[-1]) - f(W[0])), "chain": chain, "ito": chain + 0.5 * float((f2(W[:-1]) * dt).sum())}


def correlated_paths(n_paths, steps, rho, rng, years=1.0):
    """Two Brownian motions with correlation rho, built from independent ones (Cholesky): returns (t, W1, W2)."""
    t, A, dA = brownian_paths(n_paths, steps, years, rng)
    _, _, dB = brownian_paths(n_paths, steps, years, rng)
    dW2 = rho * dA + np.sqrt(1 - rho ** 2) * dB
    return t, A, np.concatenate([np.zeros((n_paths, 1)), dW2.cumsum(axis=1)], axis=1)


def realised_vol(log_returns, years):
    """Annualised volatility from log returns: the square root of their sum of squares per year."""
    return np.sqrt((np.asarray(log_returns) ** 2).sum(axis=-1) / years)

## 2. Last episode's answer

$\mathrm{Var}\big(\int_0^4 2\,dW\big) = \int_0^4 4\,dt = 16$.

In [ ]:
out["prev"] = {"answer": 16}

## 3. Taylor plus the multiplication table

For a smooth $f$, Taylor's theorem over one step gives

$$f(W_{t+\Delta t}) - f(W_t) = f'(W_t)\,\Delta W + \tfrac12 f''(W_t)\,(\Delta W)^2 + \tfrac16 f'''(W_t)\,(\Delta W)^3 + \dots$$

Add up over all steps. The first terms make the Itô integral $\int f'(W)\,dW$. In the second, the squared steps add up like time (Episode 11: $dW \cdot dW = dt$), giving $\tfrac12 \int f''(W)\,dt$. The cubes and higher are of order $(\Delta t)^{3/2}$ and vanish. So

$$df(W_t) = f'(W_t)\,dW_t + \tfrac12 f''(W_t)\,dt .$$

More generally, for $f(t, X_t)$ with $dX = \mu\,dt + \sigma\,dW$ (an **Itô process**), $(dX)^2 = \sigma^2\,dt$ and

$$df = f_t\,dt + f_x\,dX + \tfrac12 f_{xx}\,\sigma^2\,dt .$$

This is the **Itô-Doeblin formula**.

## 4. Checked on a path

On Episode 10's fine path, compare the exact change $e^{W_1} - e^{W_0}$ with the school chain rule $\sum e^{W}\Delta W$ and with Itô, $\sum e^{W}\Delta W + \tfrac12 \sum e^{W}\Delta t$, measured on coarser and finer grids. The chain rule's error doesn't shrink; Itô's does.

In [ ]:
rng = np.random.default_rng(seed)
brownian_paths(1, 1_000, 1.0, rng)                       # Episode 1's path; skip it
FINE = 2 ** 20
_, Wf, _ = brownian_paths(1, FINE, 1.0, rng)             # Episode 10's fine path
wf = Wf[0]
rows = []
for n in (16, 256, 4_096, FINE):
    c = ito_check(np.exp, np.exp, np.exp, wf[:: FINE // n])
    rows.append({"steps": n, "chain_error": c["exact"] - c["chain"], "ito_error": c["exact"] - c["ito"]})
assert abs(rows[-1]["ito_error"]) < 0.01 and abs(rows[-1]["chain_error"]) > 0.2
out["exp"] = {"rows": rows, "W_1": float(wf[-1]), "chain_fine": rows[-1]["chain_error"], "ito_fine": rows[-1]["ito_error"]}
pd.DataFrame(rows)

## 5. Episode 1, solved

For $f(x) = x^2$: $f' = 2x$, $f'' = 2$, so $d(W^2) = 2W\,dW + dt$ and

$$W_T^2 = 2\int_0^T W\,dW + T .$$

On Episode 1's path the gap was the sum of squared steps, 1.062; in the limit it is exactly $T = 1$.

In [ ]:
rng1 = np.random.default_rng(seed)
_, W1, _ = brownian_paths(1, 1_000, 1.0, rng1)
c = ito_check(lambda x: x ** 2, lambda x: 2 * x, lambda x: 2 + 0 * x, W1[0])
out["ep1"] = {"W_T_sq": c["exact"], "chain": c["chain"], "ito": c["ito"], "gap": c["exact"] - c["chain"],
              "ito_gap": c["exact"] - c["ito"]}
assert abs(out["ep1"]["ito_gap"] - (out["ep1"]["gap"] - 1)) < 1e-12        # Itô adds exactly T = 1
out["ep1"]

## 6. The log price

A share with $dS = \mu S\,dt + \sigma S\,dW$. Take $f(x) = \ln x$: $f' = 1/x$, $f'' = -1/x^2$, and $(dS)^2 = \sigma^2 S^2\,dt$, so

$$d\ln S = \big(\mu - \tfrac12 \sigma^2\big)\,dt + \sigma\,dW .$$

The log price drifts at $\mu - \tfrac12\sigma^2 = 6\%$, not $\mu = 8\%$ (Series 3, Episode 13 called it volatility drag). Simulate $S$ step by step from $dS$ itself (no formula), 100,000 paths of daily steps over a year, and average $\ln(S_1/S_0)$:

In [ ]:
mu, sig, S0 = 0.08, 0.20, 100.0
S = np.full(100_000, S0)
for _ in range(252):
    S = S * (1 + mu / 252 + sig * math.sqrt(1 / 252) * rng.standard_normal(S.size))
lr = np.log(S / S0)
out["log"] = {"mean": float(lr.mean()), "se": float(lr.std() / math.sqrt(lr.size)), "ito": mu - 0.5 * sig ** 2, "naive": mu,
              "mu": mu, "sigma": sig}
assert abs(out["log"]["mean"] - out["log"]["ito"]) < 4 * out["log"]["se"] + 0.001
out["log"]

## 7. History

Kiyosi Itô published the formula in *On a formula concerning stochastic differentials* (Nagoya Mathematical Journal 3, 1951). Wolfgang Doeblin had written down a version in his sealed envelope of 1940, opened in 2000 (Episode 1). Hence the name Itô-Doeblin.

In [ ]:
out["history"] = {"ito": 1951, "doeblin": 1940, "opened": 2000}

## 8. This episode's question

$f(x) = x^3$: $f' = 3x^2$, $f'' = 6x$, so $d(W^3) = 3W^2\,dW + 3W\,dt$. That is why $W^3 - 3\int W\,dt$ is a martingale (Episode 12).

In [ ]:
c3 = ito_check(lambda x: x ** 3, lambda x: 3 * x ** 2, lambda x: 6 * x, wf)
assert abs(c3["exact"] - c3["ito"]) < 0.01
out["question"] = {"exact": c3["exact"], "ito": c3["ito"]}

## 9. Exercises

Three to do by hand, two in code. Try each before opening its solution. The code solutions run with the rest of the notebook, so their checks are tested every time.

**Exercise 1 (By hand).** Find $d(W_t^3)$ and $d(W_t^4)$.

<details><summary>Answer</summary>

$d(W^3) = 3W^2\,dW + 3W\,dt$; $d(W^4) = 4W^3\,dW + 6W^2\,dt$.

</details>

**Exercise 2 (By hand).** Show that $Y_t = e^{\sigma W_t - \frac12\sigma^2 t}$ has $dY = \sigma Y\,dW$ (no $dt$ term), so it is a martingale (Episode 12).

<details><summary>Answer</summary>

$f(t, x) = e^{\sigma x - \frac12 \sigma^2 t}$: $f_t = -\tfrac12\sigma^2 f$, $f_x = \sigma f$, $f_{xx} = \sigma^2 f$. So $dY = -\tfrac12\sigma^2 Y dt + \sigma Y dW + \tfrac12 \sigma^2 Y dt = \sigma Y\,dW$.

</details>

**Exercise 3 (By hand).** Find $d(tW_t)$.

<details><summary>Answer</summary>

$f(t, x) = tx$: $f_t = x$, $f_x = t$, $f_{xx} = 0$, so $d(tW) = W\,dt + t\,dW$: no Itô correction, because $f$ is linear in $x$.

</details>

**Exercise 4 (In code).** Check Itô's formula pathwise for $f = \sin$ on the fine path.

In [ ]:
#@title Solution 4 (double-click to show the code) { display-mode: "form" }
cs = ito_check(np.sin, np.cos, lambda x: -np.sin(x), wf)
assert abs(cs["exact"] - cs["ito"]) < 0.01
print(f"sin: exact change {cs['exact']:+.4f}, Ito {cs['ito']:+.4f}, chain rule {cs['chain']:+.4f}")

**Exercise 5 (In code).** Repeat Section 6 with $\sigma = 40\%$. What drift does the log price have now?

In [ ]:
#@title Solution 5 (double-click to show the code) { display-mode: "form" }
S4 = np.full(50_000, S0)
r4 = np.random.default_rng(seed + 15)
for _ in range(252):
    S4 = S4 * (1 + mu / 252 + 0.4 * math.sqrt(1 / 252) * r4.standard_normal(S4.size))
m4 = float(np.log(S4 / S0).mean())
assert abs(m4 - (mu - 0.5 * 0.4 ** 2)) < 0.01
print(f"mean log return {m4:+.4f}; Ito says mu - sigma^2/2 = {mu - 0.08:+.4f}")

## 10. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")